In [7]:
# OpenDART API 사용법
import os
import requests
import pandas as pd
from datetime import datetime

# 환경변수에서 API 키 가져오기
OPENDART_API_KEY = os.getenv('OPENDART_API_KEY')
print(f"API 키: {OPENDART_API_KEY}")

# OpenDART API 기본 URL
BASE_URL = "https://opendart.fss.or.kr/api"

def get_corp_code():
    """
    고유번호(corp_code) 다운로드
    """
    url = f"{BASE_URL}/corpCode.xml"
    params = {
        'crtfc_key': OPENDART_API_KEY
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        # XML을 ZIP 파일로 받음 - 압축 해제 필요
        with open('CORPCODE.zip', 'wb') as f:
            f.write(response.content)
        print("기업 고유번호 파일 다운로드 완료: CORPCODE.zip")
    else:
        print(f"Error: {response.status_code}")

def get_company_info(corp_code):
    """
    기업개요 조회
    """
    url = f"{BASE_URL}/company.json"
    params = {
        'crtfc_key': OPENDART_API_KEY,
        'corp_code': corp_code
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Error: {response.status_code}")
        return None

def get_financial_statements(corp_code, bsns_year, reprt_code):
    """
    재무제표 조회
    corp_code: 기업 고유번호
    bsns_year: 사업연도 (2023)
    reprt_code: 보고서 코드 ('11013': 사업보고서, '11012': 반기보고서, '11014': 1분기보고서, '11011': 3분기보고서)
    """
    url = f"{BASE_URL}/fnlttSinglAcnt.json"
    params = {
        'crtfc_key': OPENDART_API_KEY,
        'corp_code': corp_code,
        'bsns_year': bsns_year,
        'reprt_code': reprt_code
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Error: {response.status_code}")
        return None

def search_company_by_name(company_name):
    """
    회사명으로 기업 검색 (고유번호 찾기)
    """
    # 실제로는 corp_code 파일을 파싱해서 검색해야 함
    # 여기서는 예시로 삼성전자의 corp_code 사용
    if "삼성전자" in company_name:
        return "00126380"  # 삼성전자 corp_code
    elif "LG전자" in company_name:
        return "00401731"  # LG전자 corp_code
    else:
        return None

# 사용 예시
print("=== OpenDART API 사용 예시 ===")

# 1. 삼성전자 기업 정보 조회
samsung_corp_code = "00126380"
company_info = get_company_info(samsung_corp_code)
if company_info:
    print("\n삼성전자 기업 정보:")
    print(f"회사명: {company_info.get('corp_name', 'N/A')}")
    print(f"CEO: {company_info.get('ceo_nm', 'N/A')}")
    print(f"설립일: {company_info.get('est_dt', 'N/A')}")
    print(f"주소: {company_info.get('adres', 'N/A')}")

# 2. 재무제표 조회 (2023년 사업보고서)
print("\n=== 재무제표 조회 ===")
financial_data = get_financial_statements(samsung_corp_code, "2023", "11013")
if financial_data and financial_data.get('status') == '000':
    df = pd.DataFrame(financial_data['list'])
    print(f"재무제표 항목 수: {len(df)}")
    print("\n주요 재무 항목:")
    # 매출액, 영업이익 등 주요 항목만 필터링
    major_items = df[df['account_nm'].str.contains('매출액|영업이익|당기순이익', na=False)]
    for _, row in major_items.head(10).iterrows():
        print(f"{row['account_nm']}: {row['thstrm_amount']:,}원")

API 키: a24aa78d95d981a1a1b0f253802f8baf47a2afed
=== OpenDART API 사용 예시 ===

삼성전자 기업 정보:
회사명: 삼성전자(주)
CEO: 전영현
설립일: 19690113
주소: 경기도 수원시 영통구  삼성로 129 (매탄동)

=== 재무제표 조회 ===
재무제표 항목 수: 28

주요 재무 항목:


ValueError: Cannot specify ',' with 's'.

# OpenDART API 사용 가이드

## 1. 환경변수 설정
- API 키가 환경변수 `OPENDART_API_KEY`에 저장되었습니다
- PowerShell 재시작 후 사용 가능합니다

## 2. 주요 API 기능

### 기업 고유번호 조회
- 모든 공시대상법인의 고유번호를 ZIP 파일로 제공
- 기업명으로 검색하기 위해 필요

### 기업개요 조회 (`company.json`)
- 기업의 기본 정보 조회
- CEO, 설립일, 주소, 업종 등

### 재무제표 조회 (`fnlttSinglAcnt.json`)
- 손익계산서, 재무상태표, 현금흐름표 등
- 연간/분기별 재무 데이터

### 보고서 코드
- `11013`: 사업보고서 (연간)
- `11012`: 반기보고서
- `11014`: 1분기보고서  
- `11011`: 3분기보고서

## 3. 사용 시 주의사항
- API 호출 제한: 분당 1,000건
- 무료 서비스이지만 회원가입 필요
- XML/JSON 형태로 데이터 제공

## 4. 추가 기능
- 공시정보 검색
- 상장기업 재무정보
- 기업지배구조 정보
- 최대주주 현황 등

In [ ]:
# 필요한 패키지 설치 (처음 한 번만 실행)
# !pip install requests pandas python-dotenv openpyxl

# .env 파일에서 API 키 로드
from dotenv import load_dotenv
import os

# .env 파일 로드
load_dotenv()

print("현재 환경변수 확인:")
OPENDART_API_KEY = os.getenv('OPENDART_API_KEY')
print(f"OPENDART_API_KEY: {OPENDART_API_KEY}")

if OPENDART_API_KEY is None:
    print("⚠️ .env 파일에서 API 키를 찾을 수 없습니다. .env 파일을 확인해주세요.")
else:
    print("✅ API 키가 성공적으로 로드되었습니다!")

현재 환경변수 확인:
OPENDART_API_KEY: a24aa78d95d981a1a1b0f253802f8baf47a2afed
✅ API 키가 성공적으로 로드되었습니다!


In [9]:
import requests
import pandas as pd
import json
import zipfile
import xml.etree.ElementTree as ET
from datetime import datetime
import time
from pathlib import Path

# 설정
BASE_URL = "https://opendart.fss.or.kr/api"
DATA_DIR = Path("./financial_data")
DATA_DIR.mkdir(exist_ok=True)

# LG 그룹 주요 계열사 정보
LG_COMPANIES = {
    "LG전자": "00401731",
    "LG화학": "00356361", 
    "LG에너지솔루션": "00164779",
    "LG디스플레이": "00413052",
    "LG유플러스": "00382713",
    "LG생활건강": "00152360",
    "LG하우시스": "00231110",
    "LG이노텍": "00165789",
    "LG CNS": "00268515",
    "LG헬로비전": "00164060"
}

def download_corp_codes():
    """기업 고유번호 파일 다운로드 및 파싱"""
    print("📥 기업 고유번호 파일 다운로드 중...")
    
    url = f"{BASE_URL}/corpCode.xml"
    params = {'crtfc_key': OPENDART_API_KEY}
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        # ZIP 파일 저장
        zip_path = DATA_DIR / "CORPCODE.zip"
        with open(zip_path, 'wb') as f:
            f.write(response.content)
        
        # ZIP 파일 압축 해제
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
        
        print("✅ 기업 고유번호 파일 다운로드 완료")
        return True
    else:
        print(f"❌ 다운로드 실패: {response.status_code}")
        return False

def parse_corp_codes():
    """XML에서 기업 정보 파싱"""
    xml_path = DATA_DIR / "CORPCODE.xml"
    
    if not xml_path.exists():
        print("❌ CORPCODE.xml 파일이 없습니다. 먼저 다운로드하세요.")
        return {}
    
    print("📊 기업 정보 파싱 중...")
    
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    corp_dict = {}
    for corp in root.findall('list'):
        corp_name = corp.find('corp_name').text
        corp_code = corp.find('corp_code').text
        corp_dict[corp_name] = corp_code
    
    print(f"✅ 총 {len(corp_dict)}개 기업 정보 파싱 완료")
    return corp_dict

def get_company_info(corp_code, company_name):
    """기업 개요 정보 조회"""
    print(f"🏢 {company_name} 기업 정보 조회 중...")
    
    url = f"{BASE_URL}/company.json"
    params = {
        'crtfc_key': OPENDART_API_KEY,
        'corp_code': corp_code
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if data.get('status') == '000':
            return data
        else:
            print(f"❌ API 오류: {data.get('message', 'Unknown error')}")
    else:
        print(f"❌ HTTP 오류: {response.status_code}")
    
    return None

def get_financial_statements_by_report(corp_code, company_name, year="2023", report_code="11013"):
    """보고서 유형별 재무제표 데이터 조회"""
    report_names = {
        '11013': '사업보고서',
        '11012': '반기보고서', 
        '11014': '1분기보고서',
        '11011': '3분기보고서'
    }
    
    print(f"💰 {company_name} {year}년 {report_names.get(report_code, '재무제표')} 조회 중...")
    
    url = f"{BASE_URL}/fnlttSinglAcnt.json"
    params = {
        'crtfc_key': OPENDART_API_KEY,
        'corp_code': corp_code,
        'bsns_year': year,
        'reprt_code': report_code
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if data.get('status') == '000':
            return data['list']
        else:
            print(f"❌ {company_name} {report_names.get(report_code)} 조회 실패: {data.get('message', 'Unknown error')}")
    else:
        print(f"❌ HTTP 오류: {response.status_code}")
    
    return None

def get_dividend_info(corp_code, company_name, year="2023"):
    """배당금 정보 조회"""
    print(f"💸 {company_name} {year}년 배당 정보 조회 중...")
    
    url = f"{BASE_URL}/alotMatter.json"
    params = {
        'crtfc_key': OPENDART_API_KEY,
        'corp_code': corp_code,
        'bsns_year': year,
        'reprt_code': '11013'
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if data.get('status') == '000':
            return data['list']
        else:
            print(f"❌ {company_name} 배당 정보 조회 실패: {data.get('message', 'Unknown error')}")
    else:
        print(f"❌ HTTP 오류: {response.status_code}")
    
    return None

def get_major_shareholder_info(corp_code, company_name, year="2023"):
    """최대주주 현황 조회"""
    print(f"👥 {company_name} {year}년 최대주주 정보 조회 중...")
    
    url = f"{BASE_URL}/hyslrSttus.json"
    params = {
        'crtfc_key': OPENDART_API_KEY,
        'corp_code': corp_code,
        'bsns_year': year,
        'reprt_code': '11013'
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        if data.get('status') == '000' and data.get('list'):
            return data['list']
        else:
            print(f"❌ {company_name} 최대주주 정보 조회 실패: {data.get('message', 'No data')}")
    else:
        print(f"❌ HTTP 오류: {response.status_code}")
    
    return None

def save_dividend_data_to_csv(dividend_data, company_name, year="2023"):
    """배당 데이터를 CSV 파일로 저장"""
    if not dividend_data:
        return False
    
    try:
        df = pd.DataFrame(dividend_data)
        
        # 파일명 생성
        safe_company_name = company_name.replace('/', '_').replace('\\', '_')
        csv_filename = f"{safe_company_name}_{year}_배당정보.csv"
        csv_path = DATA_DIR / csv_filename
        
        # CSV 저장
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        
        print(f"✅ {company_name}: 배당정보 저장 완료 ({len(df)}개 항목)")
        return True
        
    except Exception as e:
        print(f"❌ {company_name} 배당정보 저장 실패: {str(e)}")
        return False

def save_major_shareholder_to_csv(shareholder_data, company_name, year="2023"):
    """최대주주 데이터를 CSV 파일로 저장"""
    if not shareholder_data:
        return False
    
    try:
        df = pd.DataFrame(shareholder_data)
        
        # 파일명 생성
        safe_company_name = company_name.replace('/', '_').replace('\\', '_')
        csv_filename = f"{safe_company_name}_{year}_최대주주.csv"
        csv_path = DATA_DIR / csv_filename
        
        # CSV 저장
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        
        print(f"✅ {company_name}: 최대주주 정보 저장 완료 ({len(df)}개 항목)")
        return True
        
    except Exception as e:
        print(f"❌ {company_name} 최대주주 정보 저장 실패: {str(e)}")
        return False

def get_financial_statements(corp_code, company_name, year="2023"):
    """재무제표 데이터 조회 (기본 함수 - 사업보고서)"""
    return get_financial_statements_by_report(corp_code, company_name, year, '11013')

# 사용 예시
print("=== LG 그룹 계열사 재무제표 수집 시스템 ===")
print(f"📋 수집 대상: {len(LG_COMPANIES)}개 계열사")
for name in LG_COMPANIES.keys():
    print(f"  - {name}")

=== LG 그룹 계열사 재무제표 수집 시스템 ===
📋 수집 대상: 10개 계열사
  - LG전자
  - LG화학
  - LG에너지솔루션
  - LG디스플레이
  - LG유플러스
  - LG생활건강
  - LG하우시스
  - LG이노텍
  - LG CNS
  - LG헬로비전


In [10]:
def save_financial_data_to_csv(financial_data, company_name, year="2023", report_type="연간"):
    """재무제표 데이터를 CSV 파일로 저장"""
    if not financial_data:
        print(f"❌ {company_name}: 저장할 재무 데이터가 없습니다.")
        return False
    
    try:
        # DataFrame 생성
        df = pd.DataFrame(financial_data)
        
        # 주요 컬럼만 선택 (필요에 따라 조정)
        columns_to_keep = [
            'account_nm',      # 계정명
            'thstrm_nm',       # 당기명
            'thstrm_amount',   # 당기금액
            'frmtrm_nm',       # 전기명  
            'frmtrm_amount',   # 전기금액
            'fs_div',          # 재무제표구분
            'fs_nm',           # 재무제표명
            'sj_div',          # 재무제표구분명
            'account_detail'   # 계정상세
        ]
        
        # 존재하는 컬럼만 선택
        available_columns = [col for col in columns_to_keep if col in df.columns]
        df_filtered = df[available_columns]
        
        # 파일명 생성 (특수문자 제거)
        safe_company_name = company_name.replace('/', '_').replace('\\', '_')
        csv_filename = f"{safe_company_name}_{year}_{report_type}_재무제표.csv"
        csv_path = DATA_DIR / csv_filename
        
        # CSV 저장 (한글 인코딩)
        df_filtered.to_csv(csv_path, index=False, encoding='utf-8-sig')
        
        print(f"✅ {company_name}: {csv_path.name} 저장 완료 ({len(df_filtered)}개 항목)")
        
        # 주요 재무 지표 출력
        print(f"   📊 주요 재무 지표:")
        major_items = df_filtered[df_filtered['account_nm'].str.contains('매출액|영업이익|당기순이익', na=False)]
        for _, row in major_items.head(3).iterrows():
            amount = row.get('thstrm_amount', 'N/A')
            if amount and str(amount).replace(',', '').replace('-', '').isdigit():
                amount_formatted = f"{int(str(amount).replace(',', '')):,}원"
            else:
                amount_formatted = str(amount) if amount else 'N/A'
            print(f"     - {row['account_nm']}: {amount_formatted}")
        
        return True
        
    except Exception as e:
        print(f"❌ {company_name} CSV 저장 실패: {str(e)}")
        return False

def save_company_info_to_csv(company_info, company_name):
    """기업 정보를 CSV 파일로 저장"""
    if not company_info:
        return False
    
    try:
        # 기업 정보를 DataFrame으로 변환
        info_data = []
        for key, value in company_info.items():
            if key != 'status':  # status 필드 제외
                info_data.append({'항목': key, '내용': value})
        
        df_info = pd.DataFrame(info_data)
        
        # 파일명 생성
        safe_company_name = company_name.replace('/', '_').replace('\\', '_')
        csv_filename = f"{safe_company_name}_기업정보.csv"
        csv_path = DATA_DIR / csv_filename
        
        # CSV 저장
        df_info.to_csv(csv_path, index=False, encoding='utf-8-sig')
        
        print(f"✅ {company_name}: 기업정보 저장 완료")
        return True
        
    except Exception as e:
        print(f"❌ {company_name} 기업정보 저장 실패: {str(e)}")
        return False

# 메인 실행 코드
def collect_lg_financial_data():
    """LG 그룹 계열사 재무 데이터 수집 및 저장"""
    print("\n🚀 LG 그룹 재무 데이터 수집 시작!")
    print("=" * 50)
    
    # 수집할 연도 설정 (최근 3년)
    years_to_collect = ["2023", "2022", "2021"]
    
    # 성공/실패 카운터
    success_count = 0
    total_count = len(LG_COMPANIES)
    
    for company_name, corp_code in LG_COMPANIES.items():
        print(f"\n📈 [{success_count + 1}/{total_count}] {company_name} 처리 중...")
        company_success = False
        
        try:
            # 1. 기업 정보 조회 및 저장 (1회만)
            company_info = get_company_info(corp_code, company_name)
            if company_info:
                save_company_info_to_csv(company_info, company_name)
            
            # 2. 각 연도별 재무제표 및 추가 데이터 수집
            for year in years_to_collect:
                print(f"   📅 {year}년 데이터 수집 중...")
                
                # 2-1. 재무제표 조회 (사업보고서)
                financial_data = get_financial_statements(corp_code, company_name, year)
                if financial_data:
                    save_financial_data_to_csv(financial_data, company_name, year)
                    company_success = True
                
                # 2-2. 재무제표 조회 (반기보고서)
                financial_data_half = get_financial_statements_by_report(corp_code, company_name, year, '11012')
                if financial_data_half:
                    save_financial_data_to_csv(financial_data_half, company_name, year, '반기')
                
                # 2-3. 배당 정보 수집
                dividend_data = get_dividend_info(corp_code, company_name, year)
                if dividend_data:
                    save_dividend_data_to_csv(dividend_data, company_name, year)
                
                # 2-4. 최대주주 현황 수집
                major_shareholder_data = get_major_shareholder_info(corp_code, company_name, year)
                if major_shareholder_data:
                    save_major_shareholder_to_csv(major_shareholder_data, company_name, year)
                
                # API 호출 제한을 위한 대기
                time.sleep(0.2)
            
            if company_success:
                success_count += 1
            
        except Exception as e:
            print(f"❌ {company_name} 처리 중 오류 발생: {str(e)}")
        
        print("-" * 30)
    
    print(f"\n🎉 수집 완료!")
    print(f"✅ 성공: {success_count}/{total_count}개 기업")
    print(f"📅 수집 연도: {', '.join(years_to_collect)}")
    print(f"📁 저장 위치: {DATA_DIR.absolute()}")

# 실행
if OPENDART_API_KEY:
    collect_lg_financial_data()
else:
    print("❌ API 키가 설정되지 않았습니다.")


🚀 LG 그룹 재무 데이터 수집 시작!

📈 [1/10] LG전자 처리 중...
🏢 LG전자 기업 정보 조회 중...
✅ LG전자: 기업정보 저장 완료
   📅 2023년 데이터 수집 중...
💰 LG전자 2023년 사업보고서 조회 중...
✅ LG전자: LG전자_2023_연간_재무제표.csv 저장 완료 (28개 항목)
   📊 주요 재무 지표:
     - 매출액: 20,415,872,000,000원
     - 영업이익: 1,497,388,000,000원
     - 당기순이익: 546,529,000,000원
💰 LG전자 2023년 반기보고서 조회 중...
✅ LG전자: LG전자_2023_반기_재무제표.csv 저장 완료 (28개 항목)
   📊 주요 재무 지표:
     - 매출액: 19,998,457,000,000원
     - 영업이익: 741,917,000,000원
     - 당기순이익: 195,254,000,000원
💸 LG전자 2023년 배당 정보 조회 중...
✅ LG전자: 배당정보 저장 완료 (15개 항목)
👥 LG전자 2023년 최대주주 정보 조회 중...
✅ LG전자: LG전자_2023_반기_재무제표.csv 저장 완료 (28개 항목)
   📊 주요 재무 지표:
     - 매출액: 19,998,457,000,000원
     - 영업이익: 741,917,000,000원
     - 당기순이익: 195,254,000,000원
💸 LG전자 2023년 배당 정보 조회 중...
✅ LG전자: 배당정보 저장 완료 (15개 항목)
👥 LG전자 2023년 최대주주 정보 조회 중...
✅ LG전자: 최대주주 정보 저장 완료 (5개 항목)
✅ LG전자: 최대주주 정보 저장 완료 (5개 항목)
   📅 2022년 데이터 수집 중...
💰 LG전자 2022년 사업보고서 조회 중...
✅ LG전자: LG전자_2022_연간_재무제표.csv 저장 완료 (28개 항목)
   📊 주요 재무 지표:
     - 매출액: 21,111,389,000,000원
     -

In [ ]:
# 저장된 CSV 파일 분석 및 시각화
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager, rc

# 한글 폰트 설정 (Windows 환경)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def analyze_saved_data():
    """저장된 CSV 파일들을 분석"""
    print("\n📊 저장된 재무 데이터 분석")
    print("=" * 40)
    
    # 연간 재무제표 파일들만 분석 (반기 제외)
    csv_files = list(DATA_DIR.glob("*_연간_재무제표.csv"))
    
    if not csv_files:
        # 기존 형식 파일들도 체크
        csv_files = list(DATA_DIR.glob("*_재무제표.csv"))
        csv_files = [f for f in csv_files if '반기' not in f.name]
    
    if not csv_files:
        print("❌ 분석할 CSV 파일이 없습니다.")
        return
    
    print(f"📁 발견된 CSV 파일: {len(csv_files)}개")
    
    # 연도별, 회사별 데이터 수집
    combined_data = []
    years = ["2023", "2022", "2021"]
    
    for csv_file in csv_files:
        try:
            # 파일명에서 회사명과 연도 추출
            file_parts = csv_file.stem.split('_')
            company_name = file_parts[0]
            year = file_parts[1] if len(file_parts) > 1 else "unknown"
            
            if year not in years:
                continue
                
            df = pd.read_csv(csv_file, encoding='utf-8-sig')
            
            # 주요 재무 지표 추출
            revenue = df[df['account_nm'].str.contains('매출액', na=False) & 
                        df['account_nm'].str.contains('총|연결', na=False)]
            
            operating_income = df[df['account_nm'].str.contains('영업이익', na=False)]
            net_income = df[df['account_nm'].str.contains('당기순이익', na=False)]
            total_assets = df[df['account_nm'].str.contains('자산총계', na=False)]
            
            # 데이터 추출 및 저장
            revenue_val = extract_amount(revenue)
            operating_val = extract_amount(operating_income)
            net_val = extract_amount(net_income)
            assets_val = extract_amount(total_assets)
            
            if revenue_val is not None:
                combined_data.append({
                    '회사명': company_name,
                    '연도': year,
                    '매출액': revenue_val,
                    '영업이익': operating_val,
                    '당기순이익': net_val,
                    '총자산': assets_val,
                    '파일경로': str(csv_file)
                })
            
            print(f"✅ {company_name} ({year}): 분석 완료")
            
        except Exception as e:
            print(f"❌ {csv_file.name} 분석 실패: {str(e)}")
    
    if not combined_data:
        print("❌ 분석할 수 있는 데이터가 없습니다.")
        return
    
    analysis_df = pd.DataFrame(combined_data)
    
    # 연도별 매출액 상위 기업 분석
    for year in years:
        year_data = analysis_df[analysis_df['연도'] == year].copy()
        if year_data.empty:
            continue
            
        year_data = year_data.sort_values('매출액', ascending=False)
        
        print(f"\n📈 LG 그룹 계열사 매출액 순위 ({year}년)")
        print("-" * 50)
        for i, (_, row) in enumerate(year_data.iterrows(), 1):
            if pd.notna(row['매출액']):
                print(f"{i:2d}. {row['회사명']:15s}: {row['매출액']:15,.0f}원")
    
    # 3년간 매출액 추이 시각화
    create_trend_analysis(analysis_df)
    
    # 통계 요약
    print_statistics_summary(analysis_df)

def extract_amount(df_series):
    """DataFrame에서 금액 데이터 추출"""
    if df_series.empty:
        return None
    
    amount = df_series.iloc[0]['thstrm_amount']
    if amount and str(amount).replace(',', '').replace('-', '').isdigit():
        return int(str(amount).replace(',', ''))
    return None

def create_trend_analysis(df):
    """3년간 매출액 추이 분석 및 시각화"""
    try:
        # 회사별 3년간 데이터가 모두 있는 기업만 선택
        companies_with_full_data = []
        for company in df['회사명'].unique():
            company_data = df[df['회사명'] == company]
            if len(company_data) >= 2:  # 최소 2년 데이터
                companies_with_full_data.append(company)
        
        if not companies_with_full_data:
            print("❌ 추이 분석할 충분한 데이터가 없습니다.")
            return
        
        # 시각화
        plt.figure(figsize=(15, 10))
        
        # 1. 매출액 추이 그래프
        plt.subplot(2, 2, 1)
        for company in companies_with_full_data[:5]:  # 상위 5개 기업만
            company_data = df[df['회사명'] == company].sort_values('연도')
            plt.plot(company_data['연도'], company_data['매출액'], marker='o', label=company)
        
        plt.title('LG 그룹 주요 계열사 매출액 추이', fontsize=14)
        plt.xlabel('연도')
        plt.ylabel('매출액 (원)')
        plt.legend()
        plt.xticks(rotation=45)
        plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e12:.1f}조'))
        
        # 2. 2023년 매출액 비교
        plt.subplot(2, 2, 2)
        data_2023 = df[df['연도'] == '2023'].sort_values('매출액', ascending=False)
        if not data_2023.empty:
            plt.bar(data_2023['회사명'], data_2023['매출액'])
            plt.title('2023년 매출액 비교', fontsize=14)
            plt.xlabel('계열사')
            plt.ylabel('매출액 (원)')
            plt.xticks(rotation=45, ha='right')
            plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e12:.1f}조'))
        
        # 3. 영업이익률 분석 (2023년)
        plt.subplot(2, 2, 3)
        profit_data = data_2023.copy()
        profit_data = profit_data.dropna(subset=['매출액', '영업이익'])
        if not profit_data.empty:
            profit_data['영업이익률'] = (profit_data['영업이익'] / profit_data['매출액'] * 100)
            plt.bar(profit_data['회사명'], profit_data['영업이익률'])
            plt.title('2023년 영업이익률 비교', fontsize=14)
            plt.xlabel('계열사')
            plt.ylabel('영업이익률 (%)')
            plt.xticks(rotation=45, ha='right')
        
        # 4. 총자산 vs 매출액 상관관계
        plt.subplot(2, 2, 4)
        asset_data = df.dropna(subset=['매출액', '총자산'])
        if not asset_data.empty:
            plt.scatter(asset_data['총자산'], asset_data['매출액'], alpha=0.7)
            plt.title('총자산 vs 매출액 상관관계', fontsize=14)
            plt.xlabel('총자산 (원)')
            plt.ylabel('매출액 (원)')
            plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e12:.1f}조'))
            plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e12:.1f}조'))
        
        plt.tight_layout()
        plt.savefig(DATA_DIR / 'LG그룹_종합분석.png', dpi=300, bbox_inches='tight')
        plt.show()
        
    except Exception as e:
        print(f"❌ 추이 분석 실패: {str(e)}")

def print_statistics_summary(df):
    """통계 요약 출력"""
    print(f"\n📊 LG 그룹 3개년 통계 요약")
    print("=" * 50)
    
    for year in ["2023", "2022", "2021"]:
        year_data = df[df['연도'] == year]
        if year_data.empty:
            continue
            
        print(f"\n📅 {year}년:")
        print(f"  - 데이터 수집 기업: {len(year_data)}개")
        if not year_data['매출액'].isna().all():
            print(f"  - 평균 매출액: {year_data['매출액'].mean():,.0f}원")
            print(f"  - 총 매출액: {year_data['매출액'].sum():,.0f}원")
            print(f"  - 최대 매출액: {year_data['매출액'].max():,.0f}원")
    
    # 성장률 분석
    growth_analysis = []
    for company in df['회사명'].unique():
        company_data = df[df['회사명'] == company].sort_values('연도')
        if len(company_data) >= 2:
            latest_revenue = company_data.iloc[-1]['매출액']
            previous_revenue = company_data.iloc[-2]['매출액']
            if pd.notna(latest_revenue) and pd.notna(previous_revenue) and previous_revenue > 0:
                growth_rate = ((latest_revenue - previous_revenue) / previous_revenue) * 100
                growth_analysis.append({
                    '회사명': company,
                    '성장률': growth_rate,
                    '최신매출액': latest_revenue
                })
    
    if growth_analysis:
        growth_df = pd.DataFrame(growth_analysis)
        growth_df = growth_df.sort_values('성장률', ascending=False)
        
        print(f"\n📈 전년 대비 매출 성장률 상위 기업:")
        for _, row in growth_df.head(5).iterrows():
            print(f"  - {row['회사명']:12s}: {row['성장률']:+6.1f}%")

def create_summary_report():
    """요약 보고서 생성"""
    print("\n📋 요약 보고서 생성 중...")
    
    # 저장된 모든 파일 목록
    all_files = list(DATA_DIR.glob("*.csv"))
    
    # 파일 유형별 분류
    financial_files = list(DATA_DIR.glob("*_재무제표.csv"))
    company_info_files = list(DATA_DIR.glob("*_기업정보.csv"))
    dividend_files = list(DATA_DIR.glob("*_배당정보.csv"))
    shareholder_files = list(DATA_DIR.glob("*_최대주주.csv"))
    
    # 연도별 파일 수 계산
    years_data = {}
    for year in ["2023", "2022", "2021"]:
        year_files = [f for f in financial_files if year in f.name]
        years_data[year] = len(year_files)
    
    summary_data = {
        '생성일시': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        '수집_연도': ["2023", "2022", "2021"],
        '총_파일_수': len(all_files),
        '파일_유형별_집계': {
            '재무제표_파일': len(financial_files),
            '기업정보_파일': len(company_info_files),
            '배당정보_파일': len(dividend_files),
            '최대주주_파일': len(shareholder_files)
        },
        '연도별_재무제표_파일': years_data,
        '수집_대상_기업': list(LG_COMPANIES.keys()),
        '파일_목록': [f.name for f in all_files]
    }
    
    # JSON으로 요약 정보 저장
    with open(DATA_DIR / 'summary_report.json', 'w', encoding='utf-8') as f:
        json.dump(summary_data, f, ensure_ascii=False, indent=2)
    
    print(f"✅ 요약 보고서 저장: {DATA_DIR / 'summary_report.json'}")
    print(f"📊 총 {summary_data['총_파일_수']}개 파일 생성")
    print(f"📅 수집 연도: {', '.join(summary_data['수집_연도'])}")
    
    # 수집 결과 요약 출력
    print(f"\n📈 수집 결과 요약:")
    print(f"  - 재무제표: {len(financial_files)}개")
    print(f"  - 기업정보: {len(company_info_files)}개") 
    print(f"  - 배당정보: {len(dividend_files)}개")
    print(f"  - 최대주주: {len(shareholder_files)}개")
    
    for year, count in years_data.items():
        print(f"  - {year}년 재무제표: {count}개")

# 분석 실행
analyze_saved_data()
create_summary_report()


📊 저장된 재무 데이터 분석
📁 발견된 CSV 파일: 3개
✅ LG에너지솔루션 (2023): 분석 완료
✅ LG전자 (2023): 분석 완료
✅ LG화학 (2023): 분석 완료
❌ 분석할 수 있는 데이터가 없습니다.

📋 요약 보고서 생성 중...
✅ 요약 보고서 저장: financial_data\summary_report.json
📊 총 6개 파일 생성
📅 수집 연도: 2023, 2022, 2021

📈 수집 결과 요약:
  - 재무제표: 3개
  - 기업정보: 3개
  - 배당정보: 0개
  - 최대주주: 0개
  - 2023년 재무제표: 3개
  - 2022년 재무제표: 0개
  - 2021년 재무제표: 0개


# 🎯 LG 그룹 재무제표 수집 시스템 사용법 (3개년 확장판)

## 📋 실행 순서

### 1단계: 환경 설정
1. **첫 번째 셀 실행** - 필요한 패키지 설치 및 API 키 확인
   ```python
   # !pip install requests pandas python-dotenv openpyxl matplotlib seaborn
   ```

### 2단계: 데이터 수집
2. **두 번째 셀 실행** - LG 계열사 정보 및 재무제표 수집 함수 정의
3. **세 번째 셀 실행** - 실제 데이터 수집 및 CSV 저장 실행 (3개년)

### 3단계: 데이터 분석
4. **네 번째 셀 실행** - 저장된 데이터 분석 및 시각화

## 📊 생성되는 파일들 (대폭 확장!)

### CSV 파일 (연도별)
- `{회사명}_{연도}_연간_재무제표.csv`: 각 회사별 연간 상세 재무제표
- `{회사명}_{연도}_반기_재무제표.csv`: 각 회사별 반기 재무제표
- `{회사명}_{연도}_배당정보.csv`: 각 회사별 배당 현황
- `{회사명}_{연도}_최대주주.csv`: 각 회사별 최대주주 현황
- `{회사명}_기업정보.csv`: 각 회사별 기본 정보 (1회)

### 분석 결과
- `LG그룹_종합분석.png`: 3개년 추이 및 종합 분석 차트
- `summary_report.json`: 전체 수집 결과 상세 요약

## 🏢 수집 대상 계열사 (10개)
1. **LG전자** - 전자제품, 가전
2. **LG화학** - 화학, 배터리 소재
3. **LG에너지솔루션** - 배터리
4. **LG디스플레이** - 디스플레이 패널
5. **LG유플러스** - 통신서비스
6. **LG생활건강** - 화장품, 생활용품
7. **LG하우시스** - 건축자재
8. **LG이노텍** - 전자부품
9. **LG CNS** - IT 서비스
10. **LG헬로비전** - 케이블 방송

## 📈 주요 수집 데이터 (대폭 확장!)

### 재무제표 (3개년: 2021-2023)
- **연간 재무제표**: 손익계산서, 재무상태표, 현금흐름표
- **반기 재무제표**: 중간 재무 현황
- **주요 지표**: 매출액, 영업이익, 당기순이익, 총자산

### 추가 수집 데이터
- **배당 정보**: 배당금, 배당률, 배당 정책
- **최대주주 현황**: 지분율, 주요 주주 변동
- **기업 정보**: CEO, 설립일, 주소, 업종 등

### 분석 기능
- **3개년 매출 추이 분석**
- **영업이익률 비교**
- **성장률 분석**
- **총자산 vs 매출액 상관관계**

## ⚠️ 주의사항
- API 호출 제한: 분당 1,000건
- **수집 기간**: 2021-2023년 (3개년)
- 일부 계열사는 데이터가 없을 수 있음
- 한글 인코딩: UTF-8 with BOM 사용
- **예상 소요시간**: 약 5-10분 (API 호출 대기시간 포함)

## 🔧 문제 해결
- **API 키 오류**: `.env` 파일 확인
- **패키지 오류**: `pip install` 재실행
- **한글 깨짐**: 인코딩을 `utf-8-sig`로 설정
- **차트 한글 오류**: 시스템 폰트 확인
- **파일 수 급증**: 정상적인 현상 (기업당 연도별 여러 파일 생성)

## 📁 예상 파일 구조
```
financial_data/
├── LG전자_2023_연간_재무제표.csv
├── LG전자_2022_연간_재무제표.csv
├── LG전자_2021_연간_재무제표.csv
├── LG전자_2023_반기_재무제표.csv
├── LG전자_2023_배당정보.csv
├── LG전자_2023_최대주주.csv
├── LG전자_기업정보.csv
├── ... (다른 9개 계열사 동일 구조)
├── LG그룹_종합분석.png
└── summary_report.json
```

**총 예상 파일 수**: 약 150-200개 파일